In [ ]:
    !nvidia-smi

In [ ]:
import re
from sklearn.model_selection import train_test_split


In [ ]:
# Load corpus.txt
with open('/content/corpus.txt', 'r', encoding='utf-8') as f:
    corpus = [line.strip() for line in f if line.strip()]

# Check few samples and total length
print("Sample words:", corpus[:10])
print("Total words:", len(corpus))


Sample words: ['suburbanize', 'asmack', 'hypotypic', 'promoderationist', 'consonantly', 'philatelically', 'cacomelia', 'thicklips', 'luciferase', 'cinematography']
Total words: 49397


In [ ]:
from collections import defaultdict

grouped_train = defaultdict(list)
for w in corpus:
    grouped_train[len(w)].append(w)

for length, group in list(grouped_train.items())[:24]:
    print(f"Length {length}: {len(group)} words")
#prints count of first 5 word-length groups

Length 11: 5437 words
Length 6: 3716 words
Length 9: 6778 words
Length 16: 698 words
Length 14: 2019 words
Length 10: 6454 words
Length 8: 6306 words
Length 12: 4292 words
Length 13: 3094 words
Length 5: 2154 words
Length 18: 174 words
Length 4: 1077 words
Length 3: 310 words
Length 7: 5038 words
Length 15: 1226 words
Length 17: 375 words
Length 22: 8 words
Length 19: 88 words
Length 2: 70 words
Length 1: 23 words
Length 20: 40 words
Length 21: 16 words
Length 23: 3 words
Length 24: 1 words


In [ ]:
pip install hmmlearn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 7.0 MB/s eta 0:00:00


In [ ]:
from hmmlearn.hmm import CategoricalHMM
import numpy as np

# Build letter vocabulary with start/end tokens
letters = list("abcdefghijklmnopqrstuvwxyz")
tokens = ['<S>', '<E>']
vocab = tokens + letters
char_to_int = {c: i for i, c in enumerate(vocab)}
int_to_char = {i: c for c, i in char_to_int.items()}

In [ ]:
# Encode each word as integer sequence with start/end
sequences = []
for word in corpus:
    seq = [char_to_int['<S>']] + [char_to_int[ch] for ch in word if ch in letters] + [char_to_int['<E>']]
    sequences.append(seq)

In [ ]:
# Combine into numpy format
lengths = [len(seq) for seq in sequences]
X = np.concatenate([np.array(seq) for seq in sequences]).reshape(-1, 1)

print("Total sequences:", len(sequences))
print("Total observations:", len(X))
print("Sample encoded sequence:", sequences[0][:10])

Total sequences: 49397
Total observations: 570458
Sample encoded sequence: [0, 20, 22, 3, 22, 19, 3, 2, 15, 10]


In [22]:
# Train Categorical HMM
n_states = 40
model = CategoricalHMM(
    n_components=n_states,
    n_iter=50,
    tol=1e-4,
    random_state=42,
    verbose=True
)

model.fit(X, lengths)


print("HMM training complete!")
print("Number of states:", model.n_components)

         1 -1955945.46343046             +nan
         2 -1575471.80745939 +380473.65597107
         3 -1522985.83437691  +52485.97308248
         4 -1481499.23539335  +41486.59898356
         5 -1451175.56980250  +30323.66559085
         6 -1425328.51162562  +25847.05817688
         7 -1406269.51835764  +19058.99326798
         8 -1392185.40651996  +14084.11183769
         9 -1381104.57696482  +11080.82955513
        10 -1372123.01292633   +8981.56403849
        11 -1364576.99816998   +7546.01475635
        12 -1357910.17490027   +6666.82326971
        13 -1351740.76861544   +6169.40628483
        14 -1345757.31903779   +5983.44957765
        15 -1339733.33756164   +6023.98147615
        16 -1333670.84360146   +6062.49396018
        17 -1327822.90180375   +5847.94179771
        18 -1322560.23481731   +5262.66698644
        19 -1318248.11123450   +4312.12358282
        20 -1314767.26234640   +3480.84888810
        21 -1311816.26738183   +2950.99496456
        22 -1309322.86724915   +24

HMM training complete!
Number of states: 40


        50 -1281457.68960825    +609.16325248


In [ ]:
import joblib

# Save the partially trained model and mappings
joblib.dump({
    "model": model,
    "char_to_int": char_to_int,
    "int_to_char": int_to_char
}, "hmm_model_2.pkl")

